In [3]:
!pip install torchao --upgrade -q
!pip install transformers peft datasets evaluate scikit-learn accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


In [5]:
# COMPLETE SETUP — Cells 2 through 5 combined into one

# ============ IMPORTS ============
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import time
import math
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.metrics import matthews_corrcoef

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============ DATA ============
dataset = load_dataset("glue", "cola")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base", use_fast=False)

def tokenize(examples):
    return tokenizer(examples['sentence'], padding='max_length',
                     truncation=True, max_length=128)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_loader = DataLoader(tokenized['train'], batch_size=16, shuffle=True)
val_loader = DataLoader(tokenized['validation'], batch_size=16)

labels_all = np.array(tokenized['train']['labels'])
class_counts = np.bincount(labels_all)
total = len(labels_all)
class_weights = torch.tensor(
    [total / (2 * class_counts[0]), total / (2 * class_counts[1])],
    dtype=torch.float32
).to(device)
print("Class weights:", class_weights)

# ============ SoRA CLASS ============
class SoRALinear(nn.Module):
    def __init__(self, original_linear, r=8, lora_alpha=16, lora_dropout=0.1):
        super().__init__()
        in_features = original_linear.in_features
        out_features = original_linear.out_features
        self.weight = original_linear.weight
        self.bias = original_linear.bias
        self.weight.requires_grad = False
        if self.bias is not None:
            self.bias.requires_grad = False
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
        self.lora_g = nn.Parameter(torch.ones(r))
        self.r = r
        self.scaling = lora_alpha / r
        self.dropout = nn.Dropout(p=lora_dropout)
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        result = F.linear(x, self.weight, self.bias)
        lora_out = self.dropout(x) @ self.lora_A.T
        lora_out = lora_out * self.lora_g
        lora_out = lora_out @ self.lora_B.T
        return result + self.scaling * lora_out

    def proximal_step(self, lr, lambda_reg):
        with torch.no_grad():
            self.lora_g.data = (
                torch.sign(self.lora_g.data) *
                torch.clamp(torch.abs(self.lora_g.data) - lambda_reg * lr, min=0)
            )

    def active_rank(self):
        return (self.lora_g.abs() > 1e-6).sum().item()

    def effective_rank(self):
        with torch.no_grad():
            combined = self.lora_B @ torch.diag(self.lora_g) @ self.lora_A
            matrix_cpu = combined.float().cpu()
            try:
                _, sv, _ = torch.linalg.svd(matrix_cpu, full_matrices=False)
            except Exception:
                _, sv_np, _ = np.linalg.svd(matrix_cpu.detach().numpy(),
                                             full_matrices=False)
                sv = torch.tensor(sv_np)
            sv = sv.float()
            sv = sv[sv > 1e-10]
            if len(sv) == 0:
                return 0.0
            p = sv / sv.sum()
            return torch.exp(-torch.sum(p * torch.log(p))).item()

def replace_with_sora(model, target_names, r=8, lora_alpha=16, lora_dropout=0.1):
    replaced = 0
    for name, module in list(model.named_modules()):
        module_suffix = name.split('.')[-1]
        if module_suffix in target_names and isinstance(module, nn.Linear):
            parts = name.split('.')
            parent = model
            for part in parts[:-1]:
                parent = getattr(parent, part)
            sora_layer = SoRALinear(module, r=r, lora_alpha=lora_alpha,
                                    lora_dropout=lora_dropout)
            setattr(parent, parts[-1], sora_layer)
            replaced += 1
    return replaced

def apply_proximal_step(model, lr, lambda_reg):
    for module in model.modules():
        if isinstance(module, SoRALinear):
            module.proximal_step(lr, lambda_reg)

print("SoRA classes defined.")

# ============ MODEL + SoRA APPLICATION ============
base_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=2,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32
).to(device)

# Freeze entire model first
for param in base_model.parameters():
    param.requires_grad = False

# Apply SoRA
TARGET_MODULES = ["query_proj", "key_proj", "value_proj", "out_proj"]
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
LAMBDA_REG = 1e-4

n_replaced = replace_with_sora(base_model, target_names=TARGET_MODULES,
                                r=LORA_R, lora_alpha=LORA_ALPHA,
                                lora_dropout=LORA_DROPOUT)
base_model = base_model.to(device)  # re-move after SoRA injection
print(f"Replaced {n_replaced} layers with SoRALinear")

# Unfreeze only SoRA params
for module in base_model.modules():
    if isinstance(module, SoRALinear):
        module.lora_A.requires_grad = True
        module.lora_B.requires_grad = True
        module.lora_g.requires_grad = True

# Unfreeze only classifier head
for name, param in base_model.named_parameters():
    if 'classifier.weight' in name or 'classifier.bias' in name:
        param.requires_grad = True

# Verify
total_params = sum(p.numel() for p in base_model.parameters())
trainable_params = sum(p.numel() for p in base_model.parameters()
                       if p.requires_grad)
print(f"Trainable params: {trainable_params:,}")
print(f"Total params:     {total_params:,}")
print(f"Trainable %:      {100 * trainable_params / total_params:.4f}%")

# Diagnostic forward pass
sample_ids = tokenized['train'][:2]['input_ids'].to(device)
sample_mask = tokenized['train'][:2]['attention_mask'].to(device)
sample_labels = tokenized['train'][:2]['labels'].to(device)

base_model.eval()
with torch.no_grad():
    test_out = base_model(input_ids=sample_ids, attention_mask=sample_mask,
                          labels=sample_labels)
print(f"Test loss: {test_out.loss.item():.4f} | "
      f"NaN: {torch.isnan(test_out.logits).any().item()}")

model_sora = base_model
print("\nReady for training.")

Device: cuda


Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Class weights: tensor([1.6913, 0.7099], device='cuda:0')
SoRA classes defined.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

Replaced 36 layers with SoRALinear
Trainable params: 444,194
Total params:     184,866,338
Trainable %:      0.2403%
Test loss: 0.6897 | NaN: False

Ready for training.


In [6]:
model_sora = base_model  # already has SoRA applied

loss_fn = nn.CrossEntropyLoss(weight=class_weights)

# Same hyperparameters as LoRA and AdaLoRA for fair comparison
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model_sora.parameters()),
    lr=1e-4,
    weight_decay=0.01,
    eps=1e-6
)

n_epochs = 8
total_steps = len(train_loader) * n_epochs
warmup_steps = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# SoRA-specific: ℓ1 regularization strength for proximal step
# Controls sparsity — higher = more gates zeroed out faster
LAMBDA_REG = 1e-4

results_sora = {
    "epoch": [], "train_loss": [],
    "val_mcc": [], "epoch_time_sec": [],
    "avg_active_rank": [], "avg_sparsity_pct": []
}

for epoch in range(n_epochs):
    model_sora.train()
    total_loss = 0
    valid_batches = 0
    nan_count = 0
    start_time = time.time()

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model_sora(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        loss = loss_fn(outputs.logits, labels)

        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_sora.parameters(), max_norm=1.0)
        optimizer.step()

        # CRITICAL: Apply proximal step AFTER optimizer step
        # Use current LR from scheduler for correct threshold
        current_lr = optimizer.param_groups[0]['lr']
        apply_proximal_step(model_sora, current_lr, LAMBDA_REG)

        scheduler.step()

        total_loss += loss.item()
        valid_batches += 1

    # Validation
    model_sora.eval()
    all_preds, all_labels_list = [], []

    with torch.no_grad():
        for batch in val_loader:
            outputs = model_sora(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(batch['labels'].numpy())

    # Gate sparsity stats
    sora_layers = [m for m in model_sora.modules() if isinstance(m, SoRALinear)]
    active_ranks = [m.active_rank() for m in sora_layers]
    all_gates = torch.cat([m.lora_g.data.abs() for m in sora_layers])
    sparsity_pct = (all_gates < 1e-6).float().mean().item() * 100

    mcc = matthews_corrcoef(all_labels_list, all_preds)
    epoch_time = time.time() - start_time
    avg_loss = total_loss / valid_batches if valid_batches > 0 else float('nan')
    avg_active = np.mean(active_ranks)

    results_sora["epoch"].append(epoch + 1)
    results_sora["train_loss"].append(round(avg_loss, 4))
    results_sora["val_mcc"].append(round(mcc, 4))
    results_sora["epoch_time_sec"].append(round(epoch_time, 1))
    results_sora["avg_active_rank"].append(round(avg_active, 2))
    results_sora["avg_sparsity_pct"].append(round(sparsity_pct, 2))

    print(f"Epoch {epoch+1}/{n_epochs} | Loss: {avg_loss:.4f} | MCC: {mcc:.4f} | "
          f"Time: {epoch_time:.1f}s | NaN: {nan_count} | "
          f"Avg active rank: {avg_active:.1f}/8 | Sparsity: {sparsity_pct:.1f}%")

# Save immediately
with open('/content/sora_results.json', 'w') as f:
    json.dump(results_sora, f, default=str)

print(f"\n=== SoRA RESULTS ===")
print(f"Best MCC: {max(results_sora['val_mcc']):.4f}")
print(f"Avg epoch time: {np.mean(results_sora['epoch_time_sec']):.1f}s")
print(f"Final avg active rank: {results_sora['avg_active_rank'][-1]}/8")
print(f"Final sparsity: {results_sora['avg_sparsity_pct'][-1]}%")
print(results_sora)

Epoch 1/8 | Loss: 0.6972 | MCC: 0.0065 | Time: 208.5s | NaN: 0 | Avg active rank: 8.0/8 | Sparsity: 0.0%
Epoch 2/8 | Loss: 0.6964 | MCC: 0.0325 | Time: 208.4s | NaN: 0 | Avg active rank: 8.0/8 | Sparsity: 0.0%
Epoch 3/8 | Loss: 0.6924 | MCC: 0.0579 | Time: 208.2s | NaN: 0 | Avg active rank: 8.0/8 | Sparsity: 0.0%
Epoch 4/8 | Loss: 0.6904 | MCC: 0.0854 | Time: 207.5s | NaN: 0 | Avg active rank: 8.0/8 | Sparsity: 0.0%
Epoch 5/8 | Loss: 0.6843 | MCC: 0.1192 | Time: 208.6s | NaN: 0 | Avg active rank: 8.0/8 | Sparsity: 0.0%
Epoch 6/8 | Loss: 0.6753 | MCC: 0.1748 | Time: 208.2s | NaN: 0 | Avg active rank: 8.0/8 | Sparsity: 0.0%
Epoch 7/8 | Loss: 0.6699 | MCC: 0.1580 | Time: 207.4s | NaN: 0 | Avg active rank: 8.0/8 | Sparsity: 0.0%
Epoch 8/8 | Loss: 0.6659 | MCC: 0.1653 | Time: 207.9s | NaN: 0 | Avg active rank: 8.0/8 | Sparsity: 0.0%

=== SoRA RESULTS ===
Best MCC: 0.1748
Avg epoch time: 208.1s
Final avg active rank: 8.0/8
Final sparsity: 0.0%
{'epoch': [1, 2, 3, 4, 5, 6, 7, 8], 'train_loss'

In [7]:
# Run immediately after Cell 6 while model is in memory

print("=== SoRA Per-Layer Analysis ===")
sora_layers_named = [
    (name, module)
    for name, module in model_sora.named_modules()
    if isinstance(module, SoRALinear)
]

eranks_sora = []
active_ranks_sora = []
layer_data = []

for name, module in sora_layers_named:
    erank = module.effective_rank()
    active = module.active_rank()
    eranks_sora.append(erank)
    active_ranks_sora.append(active)
    layer_data.append({
        "name": name,
        "active_rank": active,
        "erank": round(erank, 3)
    })
    print(f"{name.split('.')[-1]:12s} | "
          f"Active: {active:2d}/{LORA_R} | "
          f"Eff rank: {erank:.3f} | "
          f"Gates: {module.lora_g.data.abs().round(decimals=3).tolist()[:4]}...")

# Gate value distribution
all_gates = torch.cat([m.lora_g.data.abs() for _, m in sora_layers_named])
print(f"\n=== GATE STATISTICS ===")
print(f"Total gate values: {len(all_gates)}")
print(f"Exactly zero (pruned): {(all_gates == 0.0).sum().item()}")
print(f"Near-zero (<1e-6): {(all_gates < 1e-6).sum().item()}")
print(f"Active (≥1e-6): {(all_gates >= 1e-6).sum().item()}")
print(f"Gate value distribution:")
print(f"  Mean: {all_gates.mean():.4f}")
print(f"  Std:  {all_gates.std():.4f}")
print(f"  Min:  {all_gates.min():.6f}")
print(f"  Max:  {all_gates.max():.4f}")

print(f"\n=== EFFECTIVE RANK SUMMARY ===")
print(f"Mean effective rank: {np.mean(eranks_sora):.3f}")
print(f"Min effective rank:  {np.min(eranks_sora):.3f}")
print(f"Max effective rank:  {np.max(eranks_sora):.3f}")
print(f"Max possible (r=8):  8")
print(f"Layers with active_rank < {LORA_R}: "
      f"{sum(1 for a in active_ranks_sora if a < LORA_R)}/{len(active_ranks_sora)}")

sora_summary = {
    "best_mcc": max(results_sora['val_mcc']),
    "mean_erank": round(float(np.mean(eranks_sora)), 3),
    "min_erank": round(float(np.min(eranks_sora)), 3),
    "max_erank": round(float(np.max(eranks_sora)), 3),
    "total_gates": int(len(all_gates)),
    "zero_gates": int((all_gates < 1e-6).sum().item()),
    "sparsity_pct": round(float((all_gates < 1e-6).float().mean() * 100), 2),
    "per_layer": layer_data,
    "lambda_reg": LAMBDA_REG
}

with open('/content/sora_summary.json', 'w') as f:
    json.dump(sora_summary, f)
print("\nSaved to /content/sora_summary.json")

=== SoRA Per-Layer Analysis ===
query_proj   | Active:  8/8 | Eff rank: 7.720 | Gates: [0.9990000128746033, 1.003000020980835, 0.9959999918937683, 0.9959999918937683]...
key_proj     | Active:  8/8 | Eff rank: 7.757 | Gates: [0.9919999837875366, 0.9959999918937683, 0.996999979019165, 1.0019999742507935]...
value_proj   | Active:  8/8 | Eff rank: 7.708 | Gates: [1.0069999694824219, 1.00600004196167, 1.003000020980835, 0.9950000047683716]...
query_proj   | Active:  8/8 | Eff rank: 7.680 | Gates: [1.0049999952316284, 0.996999979019165, 1.0019999742507935, 1.003999948501587]...
key_proj     | Active:  8/8 | Eff rank: 7.606 | Gates: [1.0, 1.0080000162124634, 1.003000020980835, 0.9990000128746033]...
value_proj   | Active:  8/8 | Eff rank: 7.393 | Gates: [0.9990000128746033, 1.003999948501587, 1.0069999694824219, 1.0080000162124634]...
query_proj   | Active:  8/8 | Eff rank: 7.708 | Gates: [1.00600004196167, 1.0010000467300415, 1.0019999742507935, 1.0080000162124634]...
key_proj     | Active

In [9]:
# SoRA Rerun — lambda=0.1 to demonstrate actual sparsity
# Run the full combined setup cell first if session reset, then this

LAMBDA_REG_V2 = 0.1  # 1000x stronger than before

# Fresh model + SoRA application needed — re-run setup cell if model_sora is gone
# Then run this training loop

optimizer_v2 = AdamW(
    filter(lambda p: p.requires_grad, model_sora.parameters()),
    lr=1e-4, weight_decay=0.01, eps=1e-6
)

total_steps_v2 = len(train_loader) * 3
warmup_steps_v2 = int(0.1 * total_steps_v2)
scheduler_v2 = get_linear_schedule_with_warmup(
    optimizer_v2, num_warmup_steps=warmup_steps_v2,
    num_training_steps=total_steps_v2
)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)
results_sora_v2 = {"epoch": [], "train_loss": [], "val_mcc": [],
                   "epoch_time_sec": [], "avg_active_rank": [], "sparsity_pct": []}

for epoch in range(3):
    model_sora.train()
    total_loss = 0
    valid_batches = 0
    start_time = time.time()

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer_v2.zero_grad()
        outputs = model_sora(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)

        if torch.isnan(loss) or torch.isinf(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_sora.parameters(), max_norm=1.0)
        optimizer_v2.step()

        current_lr = optimizer_v2.param_groups[0]['lr']
        apply_proximal_step(model_sora, current_lr, LAMBDA_REG_V2)
        scheduler_v2.step()

        total_loss += loss.item()
        valid_batches += 1

    model_sora.eval()
    all_preds, all_labels_list = [], []
    with torch.no_grad():
        for batch in val_loader:
            outputs = model_sora(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(batch['labels'].numpy())

    sora_layers = [m for m in model_sora.modules() if isinstance(m, SoRALinear)]
    active_ranks = [m.active_rank() for m in sora_layers]
    all_gates = torch.cat([m.lora_g.data.abs() for m in sora_layers])
    sparsity_pct = (all_gates < 1e-6).float().mean().item() * 100

    mcc = matthews_corrcoef(all_labels_list, all_preds)
    epoch_time = time.time() - start_time
    avg_loss = total_loss / valid_batches if valid_batches > 0 else float('nan')

    results_sora_v2["epoch"].append(epoch + 1)
    results_sora_v2["train_loss"].append(round(avg_loss, 4))
    results_sora_v2["val_mcc"].append(round(mcc, 4))
    results_sora_v2["epoch_time_sec"].append(round(epoch_time, 1))
    results_sora_v2["avg_active_rank"].append(round(np.mean(active_ranks), 2))
    results_sora_v2["sparsity_pct"].append(round(sparsity_pct, 2))

    print(f"Epoch {epoch+1}/3 | Loss: {avg_loss:.4f} | MCC: {mcc:.4f} | "
          f"Time: {epoch_time:.1f}s | "
          f"Avg active rank: {np.mean(active_ranks):.1f}/8 | "
          f"Sparsity: {sparsity_pct:.1f}%")

with open('/content/sora_v2_results.json', 'w') as f:
    json.dump(results_sora_v2, f, default=str)
print("\n=== SoRA v2 (lambda=0.1) RESULTS ===")
print(results_sora_v2)

Epoch 1/3 | Loss: 0.6664 | MCC: 0.1898 | Time: 211.9s | Avg active rank: 8.0/8 | Sparsity: 0.0%
Epoch 2/3 | Loss: 0.6552 | MCC: 0.2257 | Time: 208.6s | Avg active rank: 8.0/8 | Sparsity: 0.0%
Epoch 3/3 | Loss: 0.6460 | MCC: 0.2294 | Time: 207.8s | Avg active rank: 8.0/8 | Sparsity: 0.0%

=== SoRA v2 (lambda=0.1) RESULTS ===
{'epoch': [1, 2, 3], 'train_loss': [0.6664, 0.6552, 0.646], 'val_mcc': [np.float64(0.1898), np.float64(0.2257), np.float64(0.2294)], 'epoch_time_sec': [211.9, 208.6, 207.8], 'avg_active_rank': [np.float64(8.0), np.float64(8.0), np.float64(8.0)], 'sparsity_pct': [0.0, 0.0, 0.0]}
